In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, FunctionTransformer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression

from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix
)

In [38]:
# # Baseline test with clean_description only
# X_text = df["clean_description"]
# y_text = df["ticket_priority"]

# X_tr, X_te, y_tr, y_te = train_test_split(
#     X_text, y_text, test_size=0.2, random_state=42, stratify=y_text
# )

# text_pipeline = Pipeline([
#     ("tfidf", TfidfVectorizer(max_features=5000, ngram_range=(1, 2))),
#     (
#         "clf",
#         LogisticRegression(
#             max_iter=2000, class_weight="balanced", random_state=42
#         ),
#     ),
# ])

# text_pipeline.fit(X_tr, y_tr)
# print(
#     "Text-only Accuracy:",
#     accuracy_score(y_te, text_pipeline.predict(X_te)),
# )

In [39]:
df=pd.read_csv(r"D:\customer_support-ai\data\customer_tickets_cleaned.csv")
df.columns

Index(['ticket_id', 'customer_name', 'customer_email', 'product_purchased',
       'date_of_purchase', 'ticket_type', 'ticket_subject',
       'ticket_description', 'ticket_priority', 'first_response_time',
       'time_to_resolution', 'customer_satisfaction_rating',
       'clean_description', 'priority_encoded', 'satisfaction_category',
       'word_count', 'purchase_age_days', 'sentiment_score'],
      dtype='object')

In [40]:
feature_columns = [
    "date_of_purchase",
    "ticket_type",
    "ticket_subject",
    "clean_description",
    "satisfaction_category",
    "sentiment_score"
]

X = df[feature_columns].copy()

y = df["ticket_priority"]
print("X shape:", X.shape)
print("y shape:", y.shape)
y.value_counts()
y.value_counts(normalize=True) * 100

X shape: (236, 6)
y shape: (236,)


ticket_priority
High      50.000000
Medium    44.491525
Low        5.508475
Name: proportion, dtype: float64

In [41]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)
print("X_train:", X_train.shape)
print("X_test :", X_test.shape)

print("y_train:", y_train.shape)
print("y_test :", y_test.shape)

X_train: (188, 6)
X_test : (48, 6)
y_train: (188,)
y_test : (48,)


In [42]:
text_columns = [
    "clean_description",
    "ticket_subject"
]
categorical_columns = [
    "ticket_type",
    # "satisfaction_category"
]
date_column = [
    "date_of_purchase"
]
numerical_columns = [
    "sentiment_score"
]

In [43]:
def extract_date_features(X):
    
    dates = pd.Series(
        pd.to_datetime(
            np.asarray(X).ravel(),
            errors="coerce"
        )
    )

    return np.column_stack([
        dates.dt.year,
        dates.dt.month,
        dates.dt.day
    ])

In [56]:
# preprocessor = ColumnTransformer(
    
#     transformers=[
        
#         # --------------------------------
#         # 1. TF-IDF for clean_description
#         # --------------------------------
#         (
#             "clean_description_tfidf",
            
#             TfidfVectorizer(
#                 max_features=5000,
#                 ngram_range=(1, 2),
#                 min_df=1,
#                 sublinear_tf=True
#             ),
            
#             "clean_description"
#         ),
        
        
#         # --------------------------------
#         # 2. TF-IDF for ticket_subject
#         # --------------------------------
#         (
#             "subject_tfidf",
            
#             TfidfVectorizer(
#                 max_features=2000,
#                 ngram_range=(1, 2)
#             ),
            
#             "ticket_subject"
#         ),
        
        
#         # --------------------------------
#         # 3. Categorical features
#         # --------------------------------
#         (
#             "categorical",
            
#             Pipeline([
                
#                 (
#                     "imputer",
#                     SimpleImputer(
#                         strategy="most_frequent"
#                     )
#                 ),
                
#                 (
#                     "onehot",
#                     OneHotEncoder(
#                         handle_unknown="ignore"
#                     )
#                 )
#             ]),
            
#             categorical_columns
#         ),
        
        
#         # --------------------------------
#         # 4. Date features
#         # --------------------------------
#         (
#             "purchase_date",
            
#             Pipeline([
                
#                 (
#                     "extract",
#                     FunctionTransformer(
#                         extract_date_features,
#                         validate=False
#                     )
#                 ),
                
#                 (
#                     "imputer",
#                     SimpleImputer(
#                         strategy="median"
#                     )
#                 )
#             ]),
            
#             date_column
#         )
#     ],
    
#     remainder="drop"
# )


from sklearn.compose import ColumnTransformer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer, OneHotEncoder, StandardScaler

preprocessor = ColumnTransformer(
    transformers=[
        # 1. TF-IDF for clean_description
        (
            "clean_description_tfidf",
            TfidfVectorizer(
                max_features=3000,
                ngram_range=(1, 2),
                min_df=2,
                sublinear_tf=True,
            ),
            "clean_description",
        ),
        # 2. TF-IDF for ticket_subject
        (
            "subject_tfidf",
            TfidfVectorizer(
                max_features=1000, ngram_range=(1, 2), sublinear_tf=True
            ),
            "ticket_subject",
        ),
        # 3. Categorical features
        (
            "categorical",
            Pipeline([
                ("imputer", SimpleImputer(strategy="most_frequent")),
                ("onehot", OneHotEncoder(handle_unknown="ignore")),
            ]),
            categorical_columns,
        ),
        # 4. Date features
        (
            "purchase_date",
            Pipeline([
                (
                    "extract",
                    FunctionTransformer(extract_date_features, validate=False),
                ),
                ("imputer", SimpleImputer(strategy="median")),
                ("scaler", StandardScaler()),
            ]),
            date_column,
        ),
        # 5. Sentiment score
        (
            "sentiment",
            Pipeline([
                ("imputer", SimpleImputer(strategy="median")),
                ("scaler", StandardScaler()),
            ]),
            numerical_columns,
        ),
    ],
    remainder="drop",
)

In [45]:
# LOGISTIC REGRESSION - 22%
# model = Pipeline([
    
#     (
#         "preprocessor",
#         preprocessor
#     ),
    
#     (
#         "classifier",
#         LogisticRegression(
#             max_iter=2000,
#             class_weight="balanced",
#             random_state=42
#         )
#     )
# ])


#randomforest - 18%
# from sklearn.ensemble import RandomForestClassifier

# # Replace Logistic Regression with Random Forest
# model = Pipeline([
#     ("preprocessor", preprocessor),
#     (
#         "classifier",
#         RandomForestClassifier(
#             n_estimators=200,
#             max_depth=15,
#             class_weight="balanced",
#             random_state=42,
#             n_jobs=-1,
#         ),
#     ),
# ])


# 2. Linear SVC Pipeline
from sklearn.svm import LinearSVC
model = Pipeline([
    ("preprocessor", preprocessor),
    (
        "classifier",
        LinearSVC(
            C=1.0, class_weight="balanced", random_state=42, max_iter=2000
        ),
    ),
])

# Fit model
model.fit(X_train, y_train)


,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('preprocessor', ...), ('classifier', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"classes_ classes_: ndarray of shape (n_classes,)The classes labels. Only exist if the last step of the pipeline is aclassifier.","ndarray[object](3,)","['High','Low','Medium']"
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0","ndarray[object](6,)","['date_of_purchase','ticket_type','ticket_subject','clean_description', 'satisfaction_category','sentiment_score']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,6
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('clean_description_tfidf', ...), ('subject_tfidf', ...), ...]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-s

X_train
   │
   ├── clean_description
   │       ↓
   │     TF-IDF
   │
   ├── ticket_subject
   │       ↓
   │     TF-IDF
   │
   ├── ticket_type
   │       ↓
   │   One-Hot Encoding
   │
   ├── satisfaction_category
   │       ↓
   │   One-Hot Encoding
   │
   └── date_of_purchase
           ↓
      Year/Month/Day
           
           ↓
     Combined features
           ↓
   Logistic Regression
           ↓
      Ticket Priority

In [46]:
y_pred = model.predict(X_test)
print(y_pred[:10])

['High' 'Medium' 'High' 'Medium' 'Medium' 'Medium' 'Low' 'Medium' 'Medium'
 'High']


In [47]:
accuracy = accuracy_score(
    y_test,
    y_pred
)

print("Accuracy:", accuracy)
print(
    "Accuracy:",
    round(accuracy * 100, 2),
    "%"
)

Accuracy: 0.9791666666666666
Accuracy: 97.92 %


In [48]:
print(
    classification_report(
        y_test,
        y_pred,
        zero_division=0
    )
)

              precision    recall  f1-score   support

        High       1.00      0.96      0.98        24
         Low       0.75      1.00      0.86         3
      Medium       1.00      1.00      1.00        21

    accuracy                           0.98        48
   macro avg       0.92      0.99      0.95        48
weighted avg       0.98      0.98      0.98        48



In [49]:
cm = confusion_matrix(
    y_test,
    y_pred
)

print(cm)

[[23  1  0]
 [ 0  3  0]
 [ 0  0 21]]


In [50]:
print("Classes:")
print(model.named_steps["classifier"].classes_)

Classes:
['High' 'Low' 'Medium']


In [51]:
results = pd.DataFrame({
    "Actual": y_test.values,
    "Predicted": y_pred
})

results.head(20)

,Actual,Predicted
0,High,High
1,Medium,Medium
2,High,High
3,Medium,Medium
4,Medium,Medium
5,Medium,Medium
6,Low,Low
7,Medium,Medium
8,Medium,Medium
9,High,High


In [52]:
wrong_predictions = results[
    results["Actual"] != results["Predicted"]
]

wrong_predictions.head(20)


,Actual,Predicted
21,High,Low


In [53]:
# !pip install openpyxl
# import pandas as pd

# # 1. Load your dataset
# df = pd.read_csv(
#     r"D:\customer_support-ai\data\customer_tickets_cleaned.csv"
# )

# # 2. Get predictions from your trained model (replace 'model' and 'X_test' with your variables)
# y_pred = model.predict(X_test)

# # 3. Create a results DataFrame combining predictions with original descriptions
# results = pd.DataFrame(
#     {
#         "Ticket_ID": df.loc[X_test.index, "ticket_id"]
#         if "Ticket ID" in df.columns
#         else X_test.index,
#         "Description": df.loc[X_test.index, "ticket_description"],
#         "Actual": y_test,
#         "Predicted": y_pred,
#     }
# )

# # 4. Filter strictly for rows where Actual does not equal Predicted
# wrong_predictions = results[results["Actual"] != results["Predicted"]]

# # 5. Display summary and top 20 misclassifications
# print(f"Total Misclassified Tickets: {len(wrong_predictions)}")
# print(wrong_predictions.head(20))

# # 6. (Optional) Save to Excel safely using openpyxl
# wrong_predictions.to_csv(
#     "misclassified_tickets.xlsx", index=False, mode="w"
# )

In [54]:
import os
import joblib

# Save the trained components into your models folder
# joblib.dump(tfidf_vectorizer, "../models/tfidf_vectorizer.pkl")
joblib.dump(model,"LinearSVC_ticket_priority.pkl")
# If you used a standalone scaler for sentiment scores:
# joblib.dump(scaler, "../models/sentiment_scaler.pkl")

['LinearSVC_ticket_priority.pkl']